# 03 - Class steering with Gaussian/PCA denoisers

We now add the first post-hoc class control. There is no hand-written attraction
toward a coordinate. The guidance signal is the difference
between two denoisers estimated from labeled examples:

$$\Delta D = D_{target\ class} - D_{full\ data}.$$

This is the 2D counterpart of the high-noise noise-alignment stage in NA-RFM.

**Learning goals**

1. Understand a Gaussian denoiser as a noise-dependent shrinkage rule.
2. Visualize class-conditional versus unconditional denoising.
3. Apply their difference only in the high-noise sampling window.
4. Evaluate class control, target fidelity, diversity, and cost separately.


In [ ]:
import sys
from pathlib import Path

NOTES_DIR = Path.cwd()
if not (NOTES_DIR / "toy_notes.py").exists():
    NOTES_DIR = Path("notebooks/toy_data")
sys.path.insert(0, str(NOTES_DIR.resolve()))

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from toy_notes import *

SEED = 2026
set_seed(SEED)
device = choose_device()
print(environment_summary(device))


In [ ]:
model, losses, trained_now = load_or_train_model(device=device)
reference_data, reference_labels = sample_labeled_mixture(30000, device=device)
stats = estimate_gaussian_stats(reference_data, reference_labels)
TARGET_CLASS = 2
print("target class:", CLASS_NAMES[TARGET_CLASS])


## 1. Why a Gaussian denoiser makes sense at high noise

Write the base point as $x_{noise}=s\epsilon$ with
$\epsilon\sim\mathcal{N}(0,I)$ and `s=1.8`, then divide the path by `t`:

$$\frac{x_t}{t}=x_{data}+s\frac{1-t}{t}\epsilon.$$

The right side is clean data plus Gaussian noise with effective noise level
`sigma_eff=s(1-t)/t`. For a Gaussian data model, the posterior mean has a closed
form. The factor `s` matters because our base distribution is not unit variance.
At high noise the denoiser trusts coarse mean/covariance statistics; at low noise
it stays close to the observed point.

The full-data Gaussian is deliberately crude because the complete dataset is a
mixture. That limitation is useful: subtracting the full denoiser from the
target-class denoiser isolates a coarse class correction.


In [ ]:
grid_1d = torch.linspace(-4.5, 4.5, 19, device=device)
gx, gy = torch.meshgrid(grid_1d, grid_1d, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
t_value = 0.18
t = torch.full((grid.shape[0],), t_value, device=device)
delta = noise_alignment_delta(grid, t, TARGET_CLASS, stats)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.quiver(
    grid[:, 0].cpu(), grid[:, 1].cpu(), delta[:, 0].cpu(), delta[:, 1].cpu(),
    angles="xy", scale_units="xy", scale=1.7, width=0.004, color=CLASS_COLORS[TARGET_CLASS], alpha=0.8,
)
plot_labeled_points(reference_data[:2500], reference_labels[:2500], ax=ax, title=f"High-noise PCA/Gaussian correction for '{CLASS_NAMES[TARGET_CLASS]}'", alpha=0.18)
ax.set_xlim(-4.8, 4.8)
ax.set_ylim(-4.8, 4.8)
plt.show()


## 2. Paired class-steering experiment

All methods below use the same initial noise, Heun solver, step count, and target
reference. The only change is the noise-alignment strength. The correction is
active for `t in [0.04, 0.35]`, the high-noise part of our noise-to-data path.


In [ ]:
set_seed(SEED + 30)
x0 = sample_base(1800, device=device)
target_reference = sample_class(1800, TARGET_CLASS, device=device)

methods = {"baseline": base_velocity(model)}
for strength in [0.4, 0.8, 1.2, 1.6]:
    methods[f"noise alignment {strength:.1f}"] = make_noise_aligned_velocity(
        model, stats, target_class=TARGET_CLASS, strength=strength, start=0.04, end=0.35
    )

rows = []
paths = {}
for name, velocity in methods.items():
    path, seconds = timed_sample(velocity, x0, steps=64, method="heun")
    metrics = evaluate_samples(path[-1].to(device), target_reference, stats, target_class=TARGET_CLASS)
    rows.append({"method": name, **metrics, "seconds": seconds})
    paths[name] = path

noise_alignment_table = pd.DataFrame(rows)
display(noise_alignment_table.round(4))


In [ ]:
plot_trajectory_comparison(
    {"baseline": paths["baseline"], "noise alignment 1.2": paths["noise alignment 1.2"]},
    target_class=TARGET_CLASS,
)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(noise_alignment_table["method"], noise_alignment_table["target_rate"], marker="o", color="#0072B2")
axes[0].set_ylabel("target-class rate")
axes[1].plot(noise_alignment_table["method"], noise_alignment_table["target_swd"], marker="o", color="#D55E00")
axes[1].set_ylabel("SWD to target class")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Interpretation guardrails

- Higher target-class rate measures control, not sample quality.
- Lower SWD measures agreement with target examples.
- A diversity ratio far below one indicates collapse; far above one indicates
  excessive spread.
- The model remains unconditional and unchanged. Only small class/full summary
  statistics are prepared offline.
- In high-dimensional image experiments, PCA gives a low-rank covariance model.
  In 2D we keep both principal directions, so the computation is a full Gaussian
  denoiser.

**Questions**

1. Why should this coarse correction be most useful at high noise?
2. Which strength best balances target rate and target SWD in your run?
3. What information is lost when a multimodal full dataset is approximated by one Gaussian?
